# AISAM recipe generator

Use this notebook to create a `recipe.json` file for an AISAM experiment. Set `mode` to choose whether the recipe runs the full pipeline, simulation only, training only, or a sanity check.

```bash
aisam run --path /path/to/experiment_root
```

`aisam simulate` still works as a compatibility alias. If `config.json` or `simulation_params.json` also exist in the same root folder, AISAM will use those files to override recipe/default simulation values.

In [1]:
import json
from pathlib import Path

from aisam.model.defaults import default_circuit_params
from aisam.utils.pipeline import load_experiment_recipe

/Users/hossein/anaconda3/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Experiment root

Set the folder where `recipe.json` should be saved. This folder is also where AISAM will look for optional `config.json` and `simulation_params.json` files.

In [11]:
root_folder = Path("/Users/hossein/Documents/Projects/AISAM/assets/260602")
root_folder.mkdir(parents=True, exist_ok=True)
recipe_path = root_folder / "recipe.json"
recipe_path

PosixPath('/Users/hossein/Documents/Projects/AISAM/assets/260602/recipe.json')

## Required recipe fields

These fields define the experiment protocol: mode, circuit, simulator, forecaster, whether to run sanity checks, and how periodic stimulation cells should be used.

In [3]:
# Choose one: "full", "simulation", "training", or "sanity".
mode = "full"

circuit = "inverter"
simulation_model = "gillespy_tau_hybrid"

include_sanity_check = True

include_periodic_stims_in_training = False
include_periodic_stims_in_validation = True

# For mode="training", leave this as None when root_folder is an existing
# run folder containing simulation.parquet. Otherwise point to a run folder,
# simulation.parquet, or legacy simulation.pkl.
simulation_path = None

forecaster_model = {
    "type": "regressor",
    "past_feature_window": 36,
    "future_window": 12,
    "output_species": "F",
    "validation_fraction": 0.2,
    "visualization": True,
}

## Simulation settings

`total_cell` must be at least 200. If periodic stims are included in the generated simulation panel, AISAM reserves the last 100 cells for 50 red-first and 50 green-first periodic stim cells.

In [4]:
t_max = 960
sampling = 10
interval_rate = 5

total_cell = 600
num_realizations = 3
random_seed = None
progress = True

# AISAM now writes simulation.parquet by default. Keep save_pickle=True when
# you still want the legacy/debug simulation.pkl artifact for old notebooks.
save_parquet = True
save_pickle = True

noisy_sims = 0
noisy_total_cells = 350
temperatures = [0.1,0.2,0.3,0.4,0.5]

## Optional parameter overrides

Leave this dictionary empty to use AISAM's defaults for the selected circuit. Values here are written into `recipe.json`; a root-level `simulation_params.json` still has higher priority when the experiment runs.

In [5]:
default_params = default_circuit_params(circuit)
default_params

{'delta': 0.01,
 'alpha': 0.1,
 'beta': 1,
 'k': 0.4851,
 'k_tet': 5,
 'h1': 0.07100805,
 'h2': 0.0303,
 'tau_delay': 12,
 'n': 3.6,
 'n_tet': 2,
 'c2': 0.0631,
 't_max': 960,
 'sampling': 10}

In [6]:
parameter_overrides = {
    # "alpha": 1,
    # "k": 0.4851,
    # "n": 3.6,
    # "tau_delay": 12,
    # "h1": 0.07100805,
    # "h2": 0.0303,
    # "c2": 0.0631,
    # "delta": 0.01,
}

## Build and preview recipe

In [7]:
mode = mode.lower()
if mode not in {"full", "simulation", "training", "sanity"}:
    raise ValueError('mode must be one of: full, simulation, training, sanity')

recipe = {
    "mode": mode,
    "circuit": circuit,
    "simulation_model": simulation_model,
    "forecaster_model": forecaster_model,
    "include_model_training": mode in {"full", "training"},
    "include_training_simulation": mode in {"full", "simulation"},
    "include_sanity_check": include_sanity_check,
    "sanity_only": mode == "sanity",
    "include_periodic_stims_in_training": include_periodic_stims_in_training,
    "include_periodic_stims_in_validation": include_periodic_stims_in_validation,
    "t_max": t_max,
    "sampling": sampling,
    "interval_rate": interval_rate,
    "total_cell": total_cell,
    "num_realizations": num_realizations,
    "noisy_sims": noisy_sims,
    "noisy_total_cells": noisy_total_cells,
    "temperatures": temperatures,
    "progress": progress,
    "save_parquet": save_parquet,
    "save_pickle": save_pickle,
}

if simulation_path is not None:
    recipe["simulation_path"] = str(simulation_path)

if random_seed is not None:
    recipe["random_seed"] = random_seed

if parameter_overrides:
    recipe["parameters"] = parameter_overrides

print(json.dumps(recipe, indent=2))

{
  "mode": "full",
  "circuit": "inverter",
  "simulation_model": "gillespy_tau_hybrid",
  "forecaster_model": {
    "type": "regressor",
    "past_feature_window": 36,
    "future_window": 12,
    "output_species": "F",
    "validation_fraction": 0.2,
    "visualization": true
  },
  "include_model_training": true,
  "include_training_simulation": true,
  "include_sanity_check": true,
  "sanity_only": false,
  "include_periodic_stims_in_training": false,
  "include_periodic_stims_in_validation": true,
  "t_max": 960,
  "sampling": 10,
  "interval_rate": 5,
  "total_cell": 600,
  "num_realizations": 3,
  "noisy_sims": 0,
  "noisy_total_cells": 350,
  "temperatures": [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5
  ],
  "progress": true,
  "save_parquet": true,
  "save_pickle": true
}


## Save recipe.json

In [8]:
with open(recipe_path, "w") as f:
    json.dump(recipe, f, indent=2)

recipe_path

PosixPath('/Users/hossein/Documents/Projects/AISAM/assets/260602/recipe.json')

## Check how AISAM resolves the recipe

This does not run the simulation. It only previews the merged config AISAM will use after applying defaults and optional root-level files.

In [9]:
resolved = load_experiment_recipe(root_folder)
print(json.dumps(resolved["config"], indent=2, default=str))

{
  "circuit": "inverter",
  "t_max": 960,
  "sampling": 10,
  "interval_rate": 5,
  "total_cell": 600,
  "noisy_total_cells": 350,
  "num_realizations": 3,
  "noisy_sims": 0,
  "temperatures": [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5
  ],
  "progress": true,
  "mode": "full",
  "save_parquet": true,
  "save_pickle": true,
  "root_folder": "/Users/hossein/Documents/Projects/AISAM/assets/260602",
  "label": "inverter",
  "params": {
    "delta": 0.01,
    "alpha": 0.1,
    "beta": 1,
    "k": 0.4851,
    "k_tet": 5,
    "h1": 0.07100805,
    "h2": 0.0303,
    "tau_delay": 12,
    "n": 3.6,
    "n_tet": 2,
    "c2": 0.0631,
    "t_max": 960,
    "sampling": 10
  },
  "circuit_parameters": {
    "delta": 0.01,
    "alpha": 0.1,
    "beta": 1,
    "k": 0.4851,
    "k_tet": 5,
    "h1": 0.07100805,
    "h2": 0.0303,
    "tau_delay": 12,
    "n": 3.6,
    "n_tet": 2,
    "c2": 0.0631,
    "t_max": 960,
    "sampling": 10
  },
  "include_noisy": "none",
  "include_noisy_periodic": "none"